In [1]:
import glob, os, re
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

RAW_DIR = "data/raw"      # folder with the 3 DAMION xlsx files
CLEAN_DIR = "data/clean"  # folder with the 3 parquet files
OUT_DIR = "out"
os.makedirs(OUT_DIR, exist_ok=True)

In [2]:
def norm_id(s):
    return s.astype("string").str.strip().str.upper().replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA})

def norm_count(s):
    return pd.to_numeric(s, errors="coerce").astype("Int64")

def norm_date(s):
    return pd.to_datetime(s, errors="coerce").dt.normalize()

def snake(c):
    return re.sub(r"[^a-z0-9]+", "_", str(c).lower()).strip("_")

MV_CHAPTERS = r"^(89|90|90A|90B|90C)/"  # motor-vehicle statutes

In [3]:
RAW_RENAME = {
    "Docket Number": "docket", "Case ID Number": "case_id", "Race": "race", "Sex": "sex",
    "Offense Date": "offense_date", "Filing Date": "filing_date",
    "Court Location Description": "court", "Count": "count",
    "Filed Charge Code": "charge_code", "Filed Charge Description": "charge_desc",
    "Filed Charge Category": "severity", "Officer Agency Description": "agency",
    "Disp. Date": "disposition_date", "Disp. Code": "disposition_code",
    "Disposition Description": "disposition_desc", "Disp. Judge Name": "judge",
    "Case Status Description": "case_status",
}

frames = []
for f in sorted(glob.glob(os.path.join(RAW_DIR, "*Plymouth*.xlsx"))):
    d = pd.read_excel(f)
    d["file_year"] = int(re.search(r"(20\d\d)", f.split("Plymouth")[-1]).group(1))
    frames.append(d)
raw = pd.concat(frames, ignore_index=True).rename(columns=RAW_RENAME)

# drop the total row at the bottom of each file
footer = raw["docket"].isna() & raw["charge_code"].isna()
print("dropped footer rows:", footer.sum())
raw = raw[~footer].copy()

raw["docket"] = norm_id(raw["docket"])
raw["case_id"] = norm_id(raw["case_id"])
raw["count"] = norm_count(raw["count"])
for c in ["offense_date", "filing_date", "disposition_date"]:
    raw[c] = norm_date(raw[c])   # '00:00:00' junk -> NaT
raw["filing_year"] = raw["filing_date"].dt.year.astype("Int64")
raw["is_mv"] = raw["charge_code"].astype("string").str.match(MV_CHAPTERS, na=False)
raw["is_civil"] = raw["severity"].eq("CM")
raw.shape

ValueError: No objects to concatenate